# Introdução ao Apache Spark com PySpark
Primeiros passos com PySpark no Google Colab usando os dados públicos de CNPJ da Receita Federal: leitura de CSV, tipos de dados, seleções, filtros, agregações, *joins*, Spark SQL e formas de armazenamento (CSV, Parquet, ORC e particionamento).

**Origem:** curso Alura de introdução ao Spark (Spark: apresentando a ferramenta).

**Dados:** amostras dos dados públicos de CNPJ disponibilizadas pelo curso: [empresas.zip](https://caelum-online-public.s3.amazonaws.com/2273-introducao-spark/01/empresas.zip), [estabelecimentos.zip](https://caelum-online-public.s3.amazonaws.com/2273-introducao-spark/01/estabelecimentos.zip) e [socios.zip](https://caelum-online-public.s3.amazonaws.com/2273-introducao-spark/01/socios.zip) ([fonte original](https://www.gov.br/receitafederal/pt-br/assuntos/orientacao-tributaria/cadastros/consultas/dados-publicos-cnpj)). Coloque os três `.zip` na pasta `MyDrive/curso-spark` do Google Drive; o notebook extrai os arquivos.

**Índice**
1. Apache Spark e PySpark
2. Configuração do ambiente
3. Carregamento dos dados
4. Tipos de dados
5. Seleções e consultas
6. Ordenação e filtros
7. Agregações e *joins*
8. Spark SQL
9. Formas de armazenamento
10. Finalizando o Spark

In [ ]:
# Setup: Java 17 (exigido pelo Spark 4.x) e PySpark via pip
!apt-get install -qq openjdk-17-jdk-headless > /dev/null
!pip install -q pyspark

import os
import zipfile

from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import DoubleType, StringType

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

## 1. Apache Spark e PySpark
### [Apache Spark](https://spark.apache.org/)
Apache Spark é uma plataforma de computação em *cluster* que fornece uma API para programação distribuída para processamento de dados em larga escala, semelhante ao modelo *MapReduce*, mas projetada para ser rápida em consultas interativas e algoritmos iterativos.

O Spark distribui dados e tarefas em *clusters* com vários nós (imagine cada nó como um computador separado). Como cada nó processa apenas uma parte do volume total, fica mais fácil trabalhar com conjuntos de dados muito grandes. Ele é amplamente usado em preparação de dados, modelos de *machine learning* e análise de dados em tempo real.

### [PySpark](https://spark.apache.org/docs/latest/api/python/index.html)
PySpark é a interface Python para o Apache Spark: permite escrever aplicações Spark em Python e oferece o *shell* PySpark para análise interativa em ambiente distribuído. Suporta Spark SQL, DataFrame, Streaming, MLlib e Spark Core.

<center><img src="https://caelum-online-public.s3.amazonaws.com/2273-introducao-spark/01/img-001.png"/></center>

- **Spark SQL e DataFrame:** módulo para dados estruturados; fornece a abstração DataFrame e funciona como mecanismo de consulta SQL distribuído.
- **Spark Streaming:** aplicações interativas e analíticas sobre dados em *streaming* e históricos, com a tolerância a falhas do Spark.
- **MLlib:** biblioteca de aprendizado de máquina escalável, com APIs de alto nível para criar e ajustar *pipelines*.
- **Spark Core:** mecanismo de execução sobre o qual as demais funcionalidades são construídas; fornece o RDD (*Resilient Distributed Dataset*) e a computação em memória.

### Interfaces do Spark
- **RDD**: a primeira abstração do Spark; uma coleção de objetos distribuída em um *cluster*. É a API de "nível mais baixo" (Java, Python e Scala).
- **DataFrame**: conceito semelhante ao DataFrame do pandas e do R (Java, Python, R e Scala).
- **Dataset**: combina DataFrame e RDD, com a interface tipada dos RDDs (apenas Java e Scala).

Na maioria dos cenários não é preciso trabalhar com RDDs, mas vale entender a abstração: ela é a infraestrutura que torna o Spark rápido e fornece a linhagem dos dados, aparece em componentes avançados e nas visualizações da Spark UI.

## 2. Configuração do ambiente
**No Colab** (célula de setup): o original baixava uma distribuição do Spark com `wget` e usava `findspark`; aqui basta instalar o Java e o pacote `pyspark`, que já inclui o Spark para execução local.

**No Windows** ([guia de instalação](https://spark.apache.org/docs/latest/api/python/getting_started/install.html)):
1. Instale o Java (17 ou superior para o Spark 4.x) e confira com `java -version`.
2. Instale o Python 3 e confira com `python --version`.
3. Instale o PySpark com `pip install pyspark` **ou** baixe o Spark *Pre-built for Apache Hadoop 3* em [spark.apache.org/downloads](https://spark.apache.org/downloads.html) e descompacte em uma pasta (ex.: `C:\spark\`); `bin\pyspark` abre o *shell* interativo (`exit()` para sair). Com a distribuição descompactada, o pacote `findspark` ajuda o Python a localizá-la.
4. Baixe o `winutils.exe` correspondente à versão do Hadoop ([steveloughran/winutils](https://github.com/steveloughran/winutils)), copie-o para `hadoop\bin` dentro da pasta do Spark e crie as variáveis de ambiente `SPARK_HOME` (pasta do Spark) e `HADOOP_HOME` (`%SPARK_HOME%\hadoop`). Sem ele aparecem erros no console e o `spark-submit` não funciona.

### 2.1 [SparkSession](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html)
É o ponto de entrada para programar o Spark com a API de DataFrame: cria DataFrames, registra tabelas, executa SQL, faz *cache* e lê arquivos Parquet.

In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Iniciando com Spark') \
    .config('spark.ui.port', '4050') \
    .getOrCreate()

spark

A Spark UI roda na porta 4050. No Colab ela pode ser aberta, opcionalmente, com `from google.colab import output` e `output.serve_kernel_port_as_window(4050)` (o original expunha a porta com ngrok; essa parte foi removida).

## 3. Carregamento dos dados
Os arquivos não têm cabeçalho, então o Spark nomeia as colunas como `_c0`, `_c1`...; elas são renomeadas com `withColumnRenamed` a partir de uma lista com os nomes do dicionário de dados da Receita.

Referências: [SparkSession.read](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.read.html), [DataFrameReader.csv](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html).

In [ ]:
drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/curso-spark'
SAIDA = f'{PASTA}/saida'

# Extrai os .zip apenas na primeira execução
for nome in ['empresas', 'estabelecimentos', 'socios']:
    if not os.path.exists(f'{PASTA}/{nome}'):
        zipfile.ZipFile(f'{PASTA}/{nome}.zip', 'r').extractall(PASTA)

In [ ]:
def ler_csv_receita(nome, nomes_colunas):
    df = spark.read.csv(f'{PASTA}/{nome}', sep=';', inferSchema=True)
    for index, col_name in enumerate(nomes_colunas):
        df = df.withColumnRenamed(f'_c{index}', col_name)
    return df


empresasColNames = ['cnpj_basico', 'razao_social_nome_empresarial', 'natureza_juridica', 'qualificacao_do_responsavel',
                    'capital_social_da_empresa', 'porte_da_empresa', 'ente_federativo_responsavel']

estabsColNames = ['cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia',
                  'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral',
                  'nome_da_cidade_no_exterior', 'pais', 'data_de_inicio_atividade', 'cnae_fiscal_principal',
                  'cnae_fiscal_secundaria', 'tipo_de_logradouro', 'logradouro', 'numero', 'complemento', 'bairro',
                  'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_do_fax', 'fax',
                  'correio_eletronico', 'situacao_especial', 'data_da_situacao_especial']

sociosColNames = ['cnpj_basico', 'identificador_de_socio', 'nome_do_socio_ou_razao_social', 'cnpj_ou_cpf_do_socio',
                  'qualificacao_do_socio', 'data_de_entrada_sociedade', 'pais', 'representante_legal',
                  'nome_do_representante', 'qualificacao_do_representante_legal', 'faixa_etaria']

empresas = ler_csv_receita('empresas', empresasColNames)
estabelecimentos = ler_csv_receita('estabelecimentos', estabsColNames)
socios = ler_csv_receita('socios', sociosColNames)

In [ ]:
empresas.limit(5).toPandas()

In [ ]:
estabelecimentos.printSchema()

## 4. Tipos de dados
Referências: [tipos de dados](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html), [funções](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html), [withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html).

O capital social usa vírgula como separador decimal: troca-se por ponto com `regexp_replace` e converte-se para `DoubleType`.

In [ ]:
empresas = empresas.withColumn(
    'capital_social_da_empresa',
    f.regexp_replace('capital_social_da_empresa', ',', '.')
)
empresas = empresas.withColumn(
    'capital_social_da_empresa',
    empresas['capital_social_da_empresa'].cast(DoubleType())
)
empresas.printSchema()

As datas estão como inteiros `yyyyMMdd`: convertem-se para texto e depois para data com `to_date` ([padrões de data](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)).

In [ ]:
estabelecimentos = estabelecimentos \
    .withColumn(
        'data_situacao_cadastral',
        f.to_date(estabelecimentos.data_situacao_cadastral.cast(StringType()), 'yyyyMMdd')
    ) \
    .withColumn(
        'data_de_inicio_atividade',
        f.to_date(estabelecimentos.data_de_inicio_atividade.cast(StringType()), 'yyyyMMdd')
    ) \
    .withColumn(
        'data_da_situacao_especial',
        f.to_date(estabelecimentos.data_da_situacao_especial.cast(StringType()), 'yyyyMMdd')
    )

socios = socios.withColumn(
    'data_de_entrada_sociedade',
    f.to_date(socios.data_de_entrada_sociedade.cast(StringType()), 'yyyyMMdd')
)

## 5. Seleções e consultas
[DataFrame.select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html): seleciona colunas e expressões; `alias` dá nome à coluna calculada.

In [ ]:
socios \
    .select('nome_do_socio_ou_razao_social', 'faixa_etaria', f.year('data_de_entrada_sociedade').alias('ano_de_entrada')) \
    .show(5, False)

In [ ]:
# "Último nome, primeiro nome" com substring_index e concat_ws
socios \
    .select(
        f.concat_ws(
            ', ',
            f.substring_index('nome_do_socio_ou_razao_social', ' ', -1),
            f.substring_index('nome_do_socio_ou_razao_social', ' ', 1)
        ).alias('ident')
    ) \
    .show(5, False)

In [ ]:
estabelecimentos \
    .select('nome_fantasia', 'municipio',
            f.year('data_de_inicio_atividade').alias('Ano_atividade'),
            f.month('data_de_inicio_atividade').alias('Mês_atividade')) \
    .show(5, False)

In [ ]:
# Contagem de nulos em cada coluna
socios \
    .select([f.count(f.when(f.isnull(c), 1)).alias(c) for c in socios.columns]) \
    .show()

## 6. Ordenação e filtros
[orderBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html) ordena; [where](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.where.html) e [filter](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html) são equivalentes e aceitam uma expressão SQL em texto ou uma condição de coluna; [like](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.like.html) usa os curingas do SQL.

In [ ]:
socios \
    .select('nome_do_socio_ou_razao_social', 'faixa_etaria',
            f.year('data_de_entrada_sociedade').alias('ano_de_entrada')) \
    .orderBy(['ano_de_entrada', 'faixa_etaria'], ascending=[True, True]) \
    .show(10, False)

In [ ]:
empresas \
    .where('capital_social_da_empresa == 50') \
    .show(5, False)

In [ ]:
socios \
    .select('nome_do_socio_ou_razao_social') \
    .filter(socios.nome_do_socio_ou_razao_social.startswith('RODRIGO')) \
    .filter(socios.nome_do_socio_ou_razao_social.endswith('DIAS')) \
    .limit(10) \
    .toPandas()

In [ ]:
empresas \
    .select('razao_social_nome_empresarial', 'natureza_juridica', 'porte_da_empresa', 'capital_social_da_empresa') \
    .filter(f.upper(empresas['razao_social_nome_empresarial']).like('%RESTAURANTE%')) \
    .show(5, False)

## 7. Agregações e *joins*
[groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) + [agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.agg.html) agregam; [summary](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.summary.html) gera estatísticas descritivas.

Funções de agregação úteis (`pyspark.sql.functions`): `approx_count_distinct`, `avg`, `collect_list`, `collect_set`, `countDistinct`, `count`, `grouping`, `first`, `last`, `kurtosis`, `max`, `min`, `mean`, `skewness`, `stddev` (ou `stddev_samp`), `stddev_pop`, `sum`, `sum_distinct`, `variance` (ou `var_samp`) e `var_pop`.

In [ ]:
socios \
    .select(f.year('data_de_entrada_sociedade').alias('ano_de_entrada')) \
    .where('ano_de_entrada >= 2010') \
    .groupBy('ano_de_entrada') \
    .agg(f.count('*').alias('registros')) \
    .orderBy('ano_de_entrada', ascending=True) \
    .show()

In [ ]:
empresas \
    .select('cnpj_basico', 'porte_da_empresa', 'capital_social_da_empresa') \
    .groupBy('porte_da_empresa') \
    .agg(
        f.avg('capital_social_da_empresa').alias('capital_social_medio'),
        f.count('*').alias('frequencia')
    ) \
    .orderBy('porte_da_empresa', ascending=False) \
    .show()

In [ ]:
empresas \
    .select('capital_social_da_empresa') \
    .summary() \
    .show()

### 7.1 *Joins*
[DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html): cada estabelecimento recebe os dados da sua empresa pela chave `cnpj_basico`. Depois conta-se a abertura de estabelecimentos por ano e, com `union`, acrescenta-se uma linha de total (a coluna de ano vira texto para aceitar `'Total'`).

In [ ]:
empresas_join = estabelecimentos.join(empresas, 'cnpj_basico', how='left')

freq = empresas_join \
    .select('cnpj_basico', f.year('data_de_inicio_atividade').alias('data_de_inicio')) \
    .where('data_de_inicio >= 2010') \
    .groupBy('data_de_inicio') \
    .agg(f.count('cnpj_basico').alias('frequencia')) \
    .orderBy('data_de_inicio', ascending=True)

freq.limit(7).toPandas()

In [ ]:
freq \
    .withColumn('data_de_inicio', freq['data_de_inicio'].cast(StringType())) \
    .union(
        freq.select(
            f.lit('Total').alias('data_de_inicio'),
            f.sum(freq.frequencia).alias('frequencia')
        )
    ) \
    .show()

## 8. Spark SQL
[SparkSession.sql](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.sql.html) executa consultas SQL sobre *views* temporárias, criadas com `createOrReplaceTempView`. Os nomes `empresasView`, `empresasJoinView` e `freqView` existem apenas dentro do Spark (são usados nas *strings* SQL). Aspas triplas permitem escrever a consulta em várias linhas. Leitura complementar: [Spark RDDs vs DataFrames vs SparkSQL](https://community.cloudera.com/t5/Community-Articles/Spark-RDDs-vs-DataFrames-vs-SparkSQL/ta-p/246547).

In [ ]:
empresas.createOrReplaceTempView('empresasView')

spark.sql(
    'SELECT * FROM empresasView WHERE capital_social_da_empresa == 50'
).show(5, truncate=False)

In [ ]:
empresas_join.createOrReplaceTempView('empresasJoinView')

freq_sql = spark.sql("""
    SELECT
        YEAR(data_de_inicio_atividade) AS ano_atividade,
        COUNT(cnpj_basico) AS contagem,
        MEAN(cnpj_basico) AS media
    FROM empresasJoinView
    WHERE natureza_juridica >= 1000 OR cnpj_basico LIKE '1%'
    GROUP BY ano_atividade
    ORDER BY ano_atividade DESC
""")

freq_sql.show(5, truncate=False)

In [ ]:
freq_sql.createOrReplaceTempView('freqView')

spark.sql("""
    SELECT CAST(ano_atividade AS STRING) AS ano_atividade, contagem, media
    FROM freqView
    WHERE ano_atividade > 2010

    UNION ALL

    SELECT 'Total' AS ano_atividade, SUM(contagem) AS contagem, AVG(media) AS media
    FROM freqView
    WHERE ano_atividade > 2010
""").show()

## 9. Formas de armazenamento
[DataFrame.write](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.write.html) grava em vários formatos. O Spark grava uma **pasta** com um arquivo por partição; `mode='overwrite'` substitui o conteúdo existente. As saídas vão para `curso-spark/saida`, separadas das pastas de entrada.

### 9.1 CSV
[DataFrameWriter.csv](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.csv.html)

In [ ]:
empresas.write.csv(
    path=f'{SAIDA}/empresas/csv',
    mode='overwrite',
    sep=';',
    header=True
)

empresas2 = spark.read.csv(
    f'{SAIDA}/empresas/csv',
    sep=';',
    inferSchema=True,
    header=True
)

### 9.2 Parquet e ORC
[Apache Parquet](https://parquet.apache.org/) e ORC são formatos colunares e comprimidos, mais rápidos de ler que o CSV e que guardam o *schema* (tipos das colunas). [DataFrameWriter.parquet](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.parquet.html)

In [ ]:
empresas.write.parquet(
    path=f'{SAIDA}/empresas/parquet',
    mode='overwrite'
)
empresas_parquet = spark.read.parquet(f'{SAIDA}/empresas/parquet')

In [ ]:
empresas.write.orc(
    path=f'{SAIDA}/empresas/orc',
    mode='overwrite'
)
empresas_orc = spark.read.orc(f'{SAIDA}/empresas/orc')

### 9.3 Particionamento dos dados
`coalesce(n)` reduz o DataFrame a `n` partições (com `n=1`, um único arquivo); [partitionBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html) cria uma subpasta para cada valor da coluna (aqui, o porte da empresa). `%%time` mede o tempo de execução da célula.

In [ ]:
%%time
empresas.coalesce(1).write.csv(
    path=f'{SAIDA}/empresas/csv-unico',
    mode='overwrite',
    sep=';',
    header=True
)

In [ ]:
%%time
empresas.write.parquet(
    path=f'{SAIDA}/empresas/parquet-partitionBy',
    mode='overwrite',
    partitionBy='porte_da_empresa'
)

## 10. Finalizando o Spark

In [ ]:
spark.stop()